# SprayLine Service 與 API 介紹表格


## Service 介紹

| Service | 主要功能 |
|---|---|
| **Integrated Service** | 整合 `sensor_1min`、`sensor_3min` 與 `batch_run`，依照 TimeMode 或 BatchMode 查詢過去、現在與批次資料，並建立 `time_series`、`current_snapshot`、`metrics`、`components`、`viewer_state` 與站別輸出。 |
| **Event Rule／Ontology Service** | 根據 Ontology TTL 中的門檻規則判斷感測值屬於 `normal`、`warning` 或 `fault`，並提供規則來源、異常原因與處理對策；Ontology 無法使用時，改用 JSON threshold fallback。 |
| **Monitoring Service** | 依照門檻判斷設備即時狀態，產生告警事件、避免重複告警，並更新 `alert_event` 與 `batch_station_status`。 |
| **Future Service** | 以目前資料或 anchor batch 為基準進行簡易 future projection，產生 `future_time`、`future_qc`、`future_quality_score`、預測 NG 數與 `risk_level`；目前尚未使用正式 ML 模型。 |
| **Troubleshooting Service** | 根據元件、狀態、原因與處置資料，提供可能原因、處理建議、預估停機時間與所需技能。 |


## 前端／UI 使用 API

這一類 API 主要提供 Engineer UI 或 Manager UI 查詢與顯示資料，原則上不直接執行完整後端流程。

| API | 使用的 Service／資料來源 | 主要功能 |
|---|---|---|
| `POST /api/time-series` | Integrated Service | 取得完整整合查詢結果，包含時間序列、感測快照、計算指標、元件狀態與站點狀態。 |
| `POST /api/time-series/ui/summary` | Integrated Service | 取得三個站點摘要，例如正常／警告／異常站點數、平均品質分數與最高風險站點。 |
| `POST /api/time-series/ui/station-detail` | Integrated Service | 取得指定站點的感測值、品質指標、元件狀態、TimeMode／BatchMode 資訊與資料時間。 |
| `POST /api/time-series/ui/component-detail` | Integrated Service＋Event Rule＋Troubleshooting | 取得指定元件的目前數值、狀態、判斷依據、規則來源、可能原因與處理建議。 |
| `GET /api/alerts` | `alert_event` | 查詢告警列表，可依站點、狀態、確認狀態或時間範圍篩選。 |
| `GET /api/alerts/{event_id}` | `alert_event`＋Troubleshooting 資料 | 取得指定告警的詳細資訊、異常原因與處理建議。 |
| `PATCH /api/alerts/{event_id}/acknowledge` | `alert_event` | 將指定告警標記為已確認。 |
| `GET /api/alerts/causes/{cause_id}/responses` | Troubleshooting Service | 依照原因編號查詢對應的處理方法與建議。 |


## 後端／系統執行 API

這一類 API 主要用於系統狀態檢查、Service 執行、告警產生與資料回寫，通常由後端流程、測試或維護操作使用。

| API | 使用的 Service／模組 | 主要功能 |
|---|---|---|
| `GET /api/health` | FastAPI | 確認 API 是否正常啟動，並顯示目前版本與可用功能。 |
| `GET /api/database/status` | Database | 確認 PostgreSQL 連線狀態、資料表是否存在、資料筆數與最新資料時間。 |
| `GET /api/service-orchestration/status` | Integrated／Monitoring／Future／Troubleshooting | 確認各個 Service 是否可以正常使用。 |
| `POST /api/service-orchestration/integrated/query` | Integrated Service | 執行一次整合查詢，可設定是否將計算結果寫回 Database。 |
| `POST /api/service-orchestration/integrated/run-once` | Integrated Service | 執行一次完整整合計算，並將站點狀態與品質指標寫入 Database。 |
| `POST /api/service-orchestration/monitoring/run` | Monitoring Service | 執行即時監控與門檻判斷，產生告警並更新站點狀態。 |
| `POST /api/service-orchestration/future/payload` | Future Service | 建立未來預測結果，但不寫入 Database。 |
| `POST /api/service-orchestration/future/save` | Future Service | 建立未來預測結果並寫入 `future_prediction_result`。 |
| `GET /api/service-orchestration/troubleshooting/matrix` | Troubleshooting Service | 取得元件、異常狀態、可能原因與處置建議的對應表。 |
| `GET /api/service-orchestration/troubleshooting/states/{state}/recommendations` | Troubleshooting Service | 依指定狀態取得對應的故障排查與處理建議。 |
